# 3 Ways of Doing Feature Engineering

## Feature Engineering

**Feature Engineering** can be done in two main ways:

1. **Feature Cleaning & Transformation**
2. **Feature Selection**
3. **Feature Creation**

## 1. Feature Cleaning & Transformation

Improving data quality by cleaning and transforming features to make them suitable for analysis and ML.

Common techniques:

- **Handle missing data**
- **Remove duplicates**
- **Treat outliers**
- **Scaling**
- **Normalization**
- **Encoding**

## 2. Feature Selection

**Feature Selection** means choosing the most relevant features from the dataset.

It helps to:

- Improve model performance
- Reduce unnecessary features
- Reduce model complexity

### Example

Suppose we have:

`Name, Phone, Vendor, Amount, Location, Date`

We may select:

`Vendor, Amount, Location, Date`

and remove features that are not useful for the model.

## 3. Feature Creation

**Feature Creation** means generating a **new feature from existing data**.

### Example

If we have:

`Amount = 78,000`

We can create a new feature:

`High Value = Yes`

This new feature may provide additional useful information to the ML model.

## Feature Engineering Methods

**Cleaning & Transformation → Make existing features suitable**

**Feature Selection → Choose useful features**

**Feature Creation → Create new useful features**

### Easy Memory

**Clean → Select → Create**

# Feature Selection Using Correlation

## Correlation for Feature Selection

**Correlation** measures the relationship between features.

It can be used in **feature selection** to identify and remove **redundant or irrelevant features**.

## When You Should NOT Use Correlation for Feature Selection

### 1. Non-Linear Relationship

Correlation mainly measures a **linear relationship**.

If two variables have a strong non-linear relationship, correlation may show a low value even though they are actually related.

### 2. Outliers

Outliers can strongly affect the correlation value and may give a misleading result.

### 3. Categorical Variables

Standard correlation is mainly designed for numerical variables, so it should not be directly used for ordinary categorical variables.

### 4. Correlation vs Causation

**Correlation does not mean causation.**

Two variables can be related without one variable actually causing the other.

## Easy Memory

**Do not blindly use correlation when there is:**

**Non-linear relationship → Outliers → Categorical variables → Correlation ≠ Causation**

In [1]:
import pandas as pd
df=pd.read_csv(r"..\Datasets\home_prices_FE.csv")

In [2]:
df.head()

,area_sqr_ft,bedrooms,color,price_lakhs
0,3774,2,Red,216
1,1460,3,Gray,88
2,1894,4,Gray,147
3,1730,2,Blue,84
4,1695,1,Blue,77


In [3]:
df.color.unique()

<StringArray>
['Red', 'Gray', 'Blue', 'Yellow', 'Green', 'White']
Length: 6, dtype: str

In [4]:
df.color.nunique()

6

In [5]:
df["area_sqr_ft"].corr(df["price_lakhs"])

np.float64(0.9453650260417832)

In [6]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()

df["color"]=le.fit_transform(df["color"])

In [7]:
df.head()

,area_sqr_ft,bedrooms,color,price_lakhs
0,3774,2,3,216
1,1460,3,1,88
2,1894,4,1,147
3,1730,2,0,84
4,1695,1,0,77


In [8]:
co_relation=df.corr()
co_relation

,area_sqr_ft,bedrooms,color,price_lakhs
area_sqr_ft,1.000000,0.185810,0.021205,0.945365
bedrooms,0.185810,1.000000,-0.046429,0.439445
color,0.021205,-0.046429,1.000000,-0.004626
price_lakhs,0.945365,0.439445,-0.004626,1.000000


In [9]:
co_relation["price_lakhs"]

area_sqr_ft    0.945365
bedrooms       0.439445
color         -0.004626
price_lakhs    1.000000
Name: price_lakhs, dtype: float64

In [10]:
co_relation_price=abs(co_relation["price_lakhs"])
co_relation_price

area_sqr_ft    0.945365
bedrooms       0.439445
color          0.004626
price_lakhs    1.000000
Name: price_lakhs, dtype: float64

In [11]:
co_relation_price[co_relation_price > 0.2]  # mean usefull greater than 0.2

area_sqr_ft    0.945365
bedrooms       0.439445
price_lakhs    1.000000
Name: price_lakhs, dtype: float64

In [12]:
co_relation_price[co_relation_price > 0.2].index

Index(['area_sqr_ft', 'bedrooms', 'price_lakhs'], dtype='str')

In [13]:
selected_feature=co_relation_price[co_relation_price > 0.2].index.drop("price_lakhs")
selected_feature

Index(['area_sqr_ft', 'bedrooms'], dtype='str')

In [14]:
x=df[selected_feature]
y=df["price_lakhs"]

In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score ,mean_squared_error

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

model=LinearRegression()
model.fit(x_train,y_train)

y_pred=model.predict(x_test)
mse=mean_squared_error(y_test,y_pred)
r2=r2_score(y_test,y_pred)

print(f"r2:{r2} , MSE:{mse}")

r2:0.9689466488379601 , MSE:76.63332198278805


In [22]:
import numpy as np
compare=pd.DataFrame({
    "Actual":y_test,"Predicted":np.round(y_pred,1)
})

compare.head(5)

,Actual,Predicted
203,116,119.1
266,174,148.1
152,113,104.3
9,115,100.4
233,190,196.2


# Feature Selection Using Variance Inflation Factor (VIF)

## Multicollinearity and VIF

## What is Multicollinearity?

**Multicollinearity** means two or more independent features are highly related to each other.

### Example

| Experience | Age | Salary |
|---:|---:|---:|
| 2 years | 22 | ₹30K |
| 5 years | 25 | ₹45K |
| 10 years | 30 | ₹70K |

Here, **Experience and Age are related**.

**Experience ↔ Age**

Both features may provide similar information to the model.

This is called **Multicollinearity**.

### Simple Idea

**Age → Salary**

**Experience → Salary**

If **Age and Experience are highly correlated**, then even if we remove **Experience**, the model can still use **Age** to get some similar information.

However, we don't automatically remove a feature just because it is correlated. We can use **VIF, domain knowledge, and model performance** to decide.

---

## What is VIF?

**VIF = Variance Inflation Factor**

VIF is used to detect **multicollinearity** between independent features.

It tells us how much the **variance of a feature's coefficient is increased** because of its relationship with other features.

### Formula

**VIF = 1 / (1 − R²)**

Here, **R²** is calculated by taking one feature as the target and predicting it using the other features.

### Example

Suppose we take **Experience**:

**Experience = Target**

**Age + Education + Location → Predict Experience → R²**

If:

**R² = 0.9**

Then:

**VIF = 1 / (1 − 0.9)**

**VIF = 1 / 0.1**

**VIF = 10**

So, **VIF = 10**, which indicates high multicollinearity.

---

## VIF Interpretation

| VIF | Meaning |
|---:|---|
| **1** | No multicollinearity |
| **1–5** | Usually acceptable |
| **5–10** | High multicollinearity |
| **> 10** | Very high multicollinearity |

---

## Why is Multicollinearity a Problem?

It is especially important in **Linear Regression** because highly correlated features can make the model's **coefficients unstable and difficult to interpret**.

For example:

**Experience → Salary**

**Age → Salary**

If **Experience and Age** contain very similar information, the model may have difficulty deciding how much importance to give to each feature.

---

## Easy Memory Trick

**Multicollinearity = Features are too similar to each other.**

**VIF = A number used to detect multicollinearity.**

**Feature ↔ Target → Normal relationship**

**Feature ↔ Feature → High correlation → Multicollinearity → Check VIF**